# Study 905 — Residual Reversal ↩️

**Strip the factor out of weekly reversal — does the *cleaned* signal finally pay?**

Blitz, Huij, Lansdorp & Verbeek (2013) argue the classic one-week reversal (buy last
week's losers, sell its winners) mostly harvests **bid-ask bounce** and **common-factor**
moves, so it dies at the spread. Their fix: regress each name's weekly return on the
market, keep the **residual**, and reverse on *that*, on a liquid subset. We run the
self-contained version on a liquid US cross-section (2010-01-04 → 2026-06-30,
50 names), with the RAW weekly reversal placed beside it as the foil.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

A stock that dropped last week is a reversal *buy* — unless it dropped because the **whole market** dropped, in which case it is not over-sold at all, just beta doing its job. Raw reversal can't tell the two apart, so it keeps buying high-beta names into market rebounds and paying the bid-ask spread for the privilege. The **residual** reversal first removes the market move (a market-model regression) and reverses only on what's left — the genuinely idiosyncratic wobble — on liquid names where the bounce is small.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-0.38, t_nw=-0.05, raw_bps=-0.03, raw_t=-0.0, lo_bps=34.12, hi_bps=34.51, gross_sharpe=-0.01)
print('residual reversal (liquid, factor-cleaned): %+.2f bps/wk (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('raw reversal (same screen, no cleaning)    : %+.2f bps/wk (NW t = %+.2f)'
      % (R['raw_bps'], R['raw_t']))
print('  residual-loser book %+.2f bps vs residual-winner book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

residual reversal (liquid, factor-cleaned): -0.38 bps/wk (NW t = -0.05)
raw reversal (same screen, no cleaning)    : -0.03 bps/wk (NW t = -0.00)
  residual-loser book +34.12 bps vs residual-winner book +34.51 bps
  gross spread Sharpe (before cost): -0.01


## 2. Does the cleaner really clean? A live synthetic control

We plant a weekly residual mean-reversion in a seeded toy world (`edge>0`) on top of a strong common factor, and check that the **residual** detector recovers it while the **raw** detector is muddied by the factor — and that both stay *silent* on the null (`edge=0`). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from resid_reversal import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=905, n_assets=40, n_days=1600))
plan = st.synthetic_detect(data.synthetic_panel(edge=0.35, seed=905, n_assets=40, n_days=2000))
print('null world   : residual NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: residual NW t = %+.2f  vs RAW NW t = %+.2f (factor-muddied)'
      % (plan['t_nw'], plan['raw_t_nw']))

null world   : residual NW t = -0.41  (should be ~0)
planted world: residual NW t = +24.99  vs RAW NW t = +13.23 (factor-muddied)


## 3. The honest verdict — on 50 mega-caps, even the *clean* signal is flat

The residual reversal, factor-stripped and liquidity-screened exactly as the recipe asks, earns **-0.38 bps/week** with NW *t* = **-0.05** — indistinguishable from zero. The raw foil is just as dead (**-0.03** bps, *t* = -0.00); dropping the liquidity screen barely moves it (+2.31 bps, *t* = +0.32). The permutation placebo sits dead-centre (*p* = 0.51) and the two eras flip sign (+2.18 → -3.18 bps), both insignificant. The synthetic control proves the cleaner *works* (it recovers a planted residual reversal at *t* = 25, silent on 0/20 nulls), so this is a genuine **absence of edge**, not broken code: short-term reversal — residual or raw — is a small-cap / illiquid-breadth phenomenon, and 50 liquid mega-caps is exactly where it is not. **Signal: None**, **Tradability: Mirage** (costs turn the flat gross deeply negative: **-3.35 bps/wk** net at just 1 bp).